## 1. Імпорт усіх необхідних бібліотек
Для виконання цієї роботи нам знадобляться бібліотеки для роботи з масивами (`numpy`), побудови графіків (`matplotlib`), створення інтерактивних віджетів (`ipywidgets`) та цифрової фільтрації сигналів (`scipy.signal`).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy import signal
from IPython.display import display

In [2]:
t = np.linspace(0, 10, 1000)
fs = 100  

saved_noise = None
saved_noise_mean = None
saved_noise_cov = None

def harmonic_with_noise(amplitude, frequency, phase, noise_mean, noise_covariance, show_noise, show_filtered, cutoff_freq):
    global saved_noise, saved_noise_mean, saved_noise_cov
    
    if (saved_noise is None or 
        noise_mean != saved_noise_mean or 
        noise_covariance != saved_noise_cov):

        std_dev = np.sqrt(noise_covariance) if noise_covariance > 0 else 0
        saved_noise = np.random.normal(noise_mean, std_dev, len(t))
        saved_noise_mean = noise_mean
        saved_noise_cov = noise_covariance


    omega = 2 * np.pi * frequency
    pure_harmonic = amplitude * np.sin(omega * t + phase)
    

    noisy_harmonic = pure_harmonic + saved_noise


    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(t, pure_harmonic, label='Чиста гармоніка', color='blue', linewidth=2)
    
    if show_noise:
        ax.plot(t, noisy_harmonic, label='Зашумлена гармоніка', color='orange', alpha=0.5)
        
    if show_filtered:

        nyq = 0.5 * fs
        normal_cutoff = cutoff_freq / nyq
        
        if 0 < normal_cutoff < 1:
            
            b, a = signal.iirfilter(N=4, Wn=normal_cutoff, btype='lowpass', ftype='butter')
            filtered_harmonic = signal.filtfilt(b, a, noisy_harmonic)
            ax.plot(t, filtered_harmonic, label='Відфільтрована', color='green', linewidth=2, linestyle='--')
        else:
            ax.text(5, 0, "Увага: Частота зрізу занадто висока!", color='red', fontsize=12, ha='center')

    ax.set_title("Візуалізація гармоніки з шумом та фільтрацією")
    ax.set_xlabel("Час, сек")
    ax.set_ylabel("Амплітуда")
    ax.legend(loc='upper right')
    ax.grid(True, linestyle=':', alpha=0.7)
    plt.show()

amp_slider = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Амплітуда:')
freq_slider = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='Частота:')
phase_slider = widgets.FloatSlider(value=0.0, min=-np.pi, max=np.pi, step=0.1, description='Фаза:')

noise_mean_slider = widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.1, description='Шум (Середнє):', style={'description_width': 'initial'})
noise_cov_slider = widgets.FloatSlider(value=0.5, min=0.0, max=3.0, step=0.1, description='Шум (Дисперсія):', style={'description_width': 'initial'})

cutoff_slider = widgets.FloatSlider(value=3.0, min=0.5, max=25.0, step=0.5, description='Зріз фільтра:')
show_noise_cb = widgets.Checkbox(value=True, description='Показувати шум')
show_filtered_cb = widgets.Checkbox(value=True, description='Показувати відфільтровану')

reset_button = widgets.Button(description='Reset', button_style='danger', icon='refresh')

def reset_values(b):
    amp_slider.value = 1.0
    freq_slider.value = 1.0
    phase_slider.value = 0.0
    noise_mean_slider.value = 0.0
    noise_cov_slider.value = 0.5
    cutoff_slider.value = 3.0
    show_noise_cb.value = True
    show_filtered_cb.value = True

reset_button.on_click(reset_values)

out = widgets.interactive_output(harmonic_with_noise, {
    'amplitude': amp_slider,
    'frequency': freq_slider,
    'phase': phase_slider,
    'noise_mean': noise_mean_slider,
    'noise_covariance': noise_cov_slider,
    'show_noise': show_noise_cb,
    'show_filtered': show_filtered_cb,
    'cutoff_freq': cutoff_slider
})

ui = widgets.VBox([
    widgets.HTML("<b>Параметри гармоніки:</b>"),
    widgets.HBox([amp_slider, freq_slider, phase_slider]),
    widgets.HTML("<hr><b>Параметри шуму та фільтрації:</b>"),
    widgets.HBox([noise_mean_slider, noise_cov_slider, cutoff_slider]),
    widgets.HTML("<hr><b>Керування відображенням:</b>"),
    widgets.HBox([show_noise_cb, show_filtered_cb, reset_button])
])

display(ui, out)

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 1200x600 with 1 Axes>', '…